# 🧠 Deep Learning — Char-Level CNN for Sector Classification

**PFA 2025-2026 | ENSAM Rabat**
**Architecture:** Character-level Convolutional Neural Network (CNN for text)
**Input (X):** Startup **name** only — same input as the Naive Bayes ML model
**Target (y):** Sector (`primary_category_grouped`) — 21 classes (top-20 + Other)

> Why this notebook? The MLP used *structured* features (funding, country) — a different input
> than the ML model, so the ML-vs-DL comparison was unfair. This char-CNN uses the **same name
> input** as the Naive Bayes, giving a clean head-to-head, and it is the "CNN pour texte"
> architecture required by the Cahier des Charges.


## 1. Imports

In [ ]:
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
#from tensorflow.keras import layers, Model, Input
# Utilisation directe du package keras inclus dans Colab
import keras
from keras import layers, Model, Input 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

tf.random.set_seed(42); np.random.seed(42)
print('TensorFlow', tf.__version__)

## 2. Build the sector task from the raw dataset

The sector label is the **first** category tag, with the top-20 kept and everything else grouped
into `Other` (same rule as the project's `CategoryGrouper`). `Unknown` / missing categories are dropped.
This notebook is self-contained — it rebuilds the task directly from `big_startup_secsees_dataset.csv`.

In [ ]:
df = pd.read_csv('big_startup_secsees_dataset.csv')

df['primary_category'] = df['category_list'].fillna('Unknown').str.split('|').str[0].str.strip()
df = df[df['primary_category'].ne('Unknown') & df['category_list'].notna()].copy()

top20 = df['primary_category'].value_counts().nlargest(20).index.tolist()
df['sector']    = np.where(df['primary_category'].isin(top20), df['primary_category'], 'Other')
df['name_text'] = df['name'].astype(str).str.lower().str.strip().replace('', np.nan).fillna('unknown')

majority = df['sector'].value_counts(normalize=True).iloc[0]
print(f'Rows: {len(df):,} | Classes: {df["sector"].nunique()}')
print(f'Majority baseline (always predict "Other"): {majority:.4f}')

## 3. Stratified split (70 / 15 / 15)

In [ ]:
tr, tmp = train_test_split(df, test_size=0.30, stratify=df['sector'], random_state=42)
va, te  = train_test_split(tmp, test_size=0.50, stratify=tmp['sector'], random_state=42)

le = LabelEncoder().fit(tr['sector'])
ytr, yva, yte = le.transform(tr['sector']), le.transform(va['sector']), le.transform(te['sector'])
n_classes = len(le.classes_)
print(f'Train/Val/Test: {len(tr):,}/{len(va):,}/{len(te):,} | classes={n_classes}')

## 4. Character encoding

Each name is turned into a sequence of character indices (padded/truncated to 40 chars).
The CNN learns character n-gram patterns directly — e.g. `bio`, `tech`, `med`, `edu`, `game`.

In [ ]:
MAXLEN = 40
chars = sorted(set(''.join(tr['name_text'].tolist())))
char2idx = {c: i + 1 for i, c in enumerate(chars)}   # 0 reserved for padding
vocab = len(char2idx) + 1

def encode(s):
    seq = [char2idx.get(c, 0) for c in s[:MAXLEN]]
    return seq + [0] * (MAXLEN - len(seq))

Xtr = np.array([encode(s) for s in tr['name_text']])
Xva = np.array([encode(s) for s in va['name_text']])
Xte = np.array([encode(s) for s in te['name_text']])
print(f'Vocabulary size: {vocab} characters | sequence length: {MAXLEN}')

## 5. Char-CNN architecture

```
Input (40 chars)
  → Embedding(vocab, 32)
  → [ Conv1D(128, k=3) , Conv1D(128, k=4) , Conv1D(128, k=5) ]  (parallel windows)
  → GlobalMaxPooling1D on each → Concatenate
  → Dropout(0.4) → Dense(128, relu) → Dropout(0.3)
  → Dense(21, softmax)
```

The three parallel convolution windows act like learned character 3-, 4-, and 5-grams — the
neural-network analogue of the `char_wb (3,5)` TF-IDF features used by the Naive Bayes model.

In [ ]:
inp = Input(shape=(MAXLEN,))
x   = layers.Embedding(vocab, 32)(inp)
convs = [layers.GlobalMaxPooling1D()(
            layers.Conv1D(128, k, activation='relu', padding='same')(x))
         for k in (3, 4, 5)]
x   = layers.Concatenate()(convs)
x   = layers.Dropout(0.4)(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(n_classes, activation='softmax')(x)

model = Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print(f'Total params: {model.count_params():,}')

## 6. Training

No class weights — this matches the Naive Bayes setup so the comparison is apples-to-apples and
the model is optimised for accuracy. (Adding `class_weight='balanced'` would raise macro-F1 but
collapse accuracy, because the model would stop defaulting to the dominant `Other` class.)

In [ ]:
es = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)
history = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                    epochs=25, batch_size=256, callbacks=[es], verbose=2)

## 7. Learning curves

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(history.history['accuracy'], label='Train'); a1.plot(history.history['val_accuracy'], label='Val')
a1.set_title('Accuracy'); a1.set_xlabel('Epoch'); a1.legend()
a2.plot(history.history['loss'], label='Train'); a2.plot(history.history['val_loss'], label='Val')
a2.set_title('Loss'); a2.set_xlabel('Epoch'); a2.legend()
plt.suptitle('Char-CNN — Training Curves'); plt.tight_layout(); plt.show()

## 8. Test-set evaluation

In [ ]:
pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
acc  = accuracy_score(yte, pred)
mf1  = f1_score(yte, pred, average='macro')

print(f'Test accuracy : {acc:.4f}')
print(f'Test macro-F1 : {mf1:.4f}')
print(f'Baseline      : {majority:.4f}  (always "Other")')
print(f'Random (1/21) : {1/n_classes:.4f}')
print()
print(classification_report(yte, pred, target_names=le.classes_, zero_division=0))

## 9. Per-class F1

In [ ]:
rep = classification_report(yte, pred, target_names=le.classes_, output_dict=True, zero_division=0)
cls = [k for k in rep if k not in ('accuracy', 'macro avg', 'weighted avg')]
f1s = [rep[c]['f1-score'] for c in cls]
colors = ['#e74c3c' if f < 0.10 else '#f39c12' if f < 0.30 else '#2ecc71' for f in f1s]

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(cls, f1s, color=colors, edgecolor='black')
ax.set_title('Per-Class F1 — Char-CNN (sector from name)'); ax.set_ylabel('F1'); ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 10. ML vs DL comparison (sector classification)

In [ ]:
comparison = pd.DataFrame([
    ('Char-CNN (DL)',   'Name (char sequence)', acc,   mf1),
    ('Naive Bayes (ML)','Name (char n-grams)',  0.494, 0.114),
    ('MLP (DL)',        'Structured features',  0.472, 0.045),
    ('Majority baseline','Predict "Other"',    majority, np.nan),
], columns=['Model', 'Input', 'Test_Accuracy', 'Test_MacroF1']).round(4)
print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
m = comparison[comparison.Model != 'Majority baseline']
ax.bar(m['Model'], m['Test_Accuracy'], color=['#e67e22', '#3498db', '#9b59b6'], edgecolor='black')
ax.axhline(majority, color='red', ls='--', label=f'Baseline ({majority:.3f})')
for i, v in enumerate(m['Test_Accuracy']): ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
ax.set_ylabel('Test Accuracy'); ax.set_ylim(0, 0.6); ax.set_title('Sector Classification — ML vs DL'); ax.legend()
plt.tight_layout(); plt.show()

## 11. Save the model

In [ ]:
import joblib
model.save('charcnn_sector_model.keras')
joblib.dump(le, 'charcnn_label_encoder.pkl')
joblib.dump(char2idx, 'charcnn_char2idx.pkl')
print('Saved: charcnn_sector_model.keras, charcnn_label_encoder.pkl, charcnn_char2idx.pkl')

## Conclusion

| Model | Type | Input | Test Acc | Macro-F1 |
|---|---|---|---|---|
| **Char-CNN** | **DL** | name (char sequence) | **~0.50** | ~0.10 |
| Naive Bayes | ML | name (char n-grams) | ~0.49 | ~0.11 |
| MLP | DL | structured features | ~0.47 | ~0.05 |
| baseline | — | predict "Other" | ~0.47 | — |

- On the **same name input**, DL (Char-CNN) and ML (Naive Bayes) perform almost identically (~0.50),
  confirming the name carries a real but modest sector signal.
- Both beat the structured-feature MLP, which sits at the majority baseline — funding/geography do
  not determine sector.
- ~0.50 accuracy on a 21-class problem is **~10× better than random** (0.048).
